In [1]:
import os

import pandas as pd

In [ ]:
#!wget https://www.mgc.ac.cn/VFs/Down/VFDB_setB_pro.fas.gz

In [ ]:
#/home/dcm/diamond makedb --in /home/dcm/bioinformatic_scripts/blast_databases/VFDB.fasta -d /home/dcm/bioinformatic_scripts/blast_databases/VFDB

In [ ]:
os.makedirs('VFDB_diamond', exist_ok=True)

final_df = pd.DataFrame()

project_list = [
    '/home/dcm/250513Pat',
    '/home/dcm/250728Pat',
    '/home/dcm/250930Pat'
    ]

for project in project_list:

    file_df = pd.read_csv(f'{project}/files.txt', sep='\t')
    
    for folder, file in zip(file_df['folder'], file_df['file']):

        os.system(f'/home/dcm/diamond blastp \
        -d /home/dcm/bioinformatic_scripts/blast_databases/VFDB.dmnd \
        -q {project}/annotation_output/protein_annotations/annotations_{file}_scaffolds_1000bp.feature_protein.fasta \
        -o VFDB_diamond/{file}_VFDB_out.txt \
        -f 6 qseqid qlen sseqid slen qseq sseq evalue bitscore pident qcovhsp salltitles \
        -k 1')

In [3]:
df_contigs = pd.read_csv('/home/dcm/250513Pat_250728Pat_250930Pat/nt_prok_blastn_out/nt_prok_blastn_out_taxids.txt', sep = '\t')

In [6]:
project_list = [
    '/home/dcm/250513Pat',
    '/home/dcm/250728Pat',
    '/home/dcm/250930Pat'
    ]

for project in project_list:

    file_df = pd.read_csv(f'{project}/files.txt', sep='\t')
    
    for folder, file in zip(file_df['folder'], file_df['file']):

        df_anno = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/annotation_out/annotations_{file}_filter.txt', sep = '\t')

        col_names = ['qseqid','qlen','sseqid','slen','qseq','sseq','evalue','bitscore','pident','qcovhsp','salltitles']
        
        df_vf = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/VFDB_diamond/{file}_VFDB_out.txt', sep = '\t', header = None, names = col_names)
        
        dfm = pd.merge(df_anno, df_vf, left_on = 'feature_id', right_on = 'qseqid', how = 'left')

        dfm = pd.merge(dfm, df_contigs, left_on = 'contig_id', right_on = 'qseqid', how = 'left')

        dfm.to_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/VFDB_diamond/{file}_VFDB_out_meta.txt', sep = '\t', index=False)


In [10]:
#make pivot table from bv-brc pgfam
dfm = pd.DataFrame()

project_list = [
    '/home/dcm/250513Pat',
    '/home/dcm/250728Pat',
    '/home/dcm/250930Pat'
    ]

for project in project_list:

    file_df = pd.read_csv(f'{project}/files.txt', sep='\t')
    
    for folder, file in zip(file_df['folder'], file_df['file']):

        df = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/VFDB_diamond/{file}_VFDB_out_meta.txt', sep = '\t', usecols = ['pgfam', 'file'])
        df = df.drop_duplicates()
        dfm = pd.concat([dfm, df], ignore_index=True)      

dfm = dfm.dropna()
dfm

pivot_df = dfm.pivot_table(index='pgfam', columns='file', aggfunc='size', fill_value=0)
pivot_df = pivot_df.reset_index()
pivot_df.to_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/pgfam.txt', sep = '\t', index=False)

In [2]:
stats_df = pd.read_csv('stats_out/fisher_IPS_20_vs_26_results.txt', sep = '\t')
stats_df = stats_df[stats_df['P-value'] < 0.05]


dfm = pd.DataFrame()

project_list = [
    '/home/dcm/250513Pat',
    '/home/dcm/250728Pat',
    '/home/dcm/250930Pat'
    ]

for project in project_list:

    file_df = pd.read_csv(f'{project}/files.txt', sep='\t')
    
    for folder, file in zip(file_df['folder'], file_df['file']):

        df_IPS = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/IPS_out/{file}_IPS_filter.txt', sep = '\t', usecols = ['A','L'])

        mask_IPS = df_IPS['L'].isin(stats_df['otu'])
        
        # Apply the mask to dfA to filter the rows
        df_IPS = df_IPS[mask_IPS]

        df_IPS = df_IPS.drop_duplicates()

        
        df = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/VFDB_diamond/{file}_VFDB_out_meta.txt', sep = '\t')

        mask = df['feature_id'].isin(df_IPS['A'])
        
        # Apply the mask to dfA to filter the rows
        df = df[mask]

        
        dfm = pd.concat([dfm, df], ignore_index=True)   

dfm.to_csv('signif_IPS_annotations.txt', sep = '\t', index = False)   

/tmp/ipykernel_1745345/2922727399.py:29: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/VFDB_diamond/{file}_VFDB_out_meta.txt', sep = '\t')
/tmp/ipykernel_1745345/2922727399.py:29: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/VFDB_diamond/{file}_VFDB_out_meta.txt', sep = '\t')
/tmp/ipykernel_1745345/2922727399.py:29: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/VFDB_diamond/{file}_VFDB_out_meta.txt', sep = '\t')
/tmp/ipykernel_1745345/2922727399.py:29: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'/home/dcm/250513Pat_250728Pat_250930Pat/VFDB_diamond/{file}_VFDB_out_met